# 02 (trial) — TweakReg Alignment

Diagnostic alignment notebook for quasars with problematic WCS solutions.
Runs TweakReg against reference catalogs with `updatehdr=False` so you can
inspect residuals before committing any WCS changes to the FLT headers.

**Input:** FLT files in `data/raw/<target>/`  
**Reference catalogs:** supply `.cat` files alongside this notebook  
**Output:** shift files written to `notebooks/trial/` for inspection

In [ ]:
import os
import yaml
import glob
from pathlib import Path
from drizzlepac import tweakreg
from IPython.display import clear_output

In [ ]:
RAW_DIR      = Path('../data/raw')
PROC_DIR     = Path('../data/processed')
TWEAKREG_DIR = Path('../data/tweakreg')
TWEAKREG_DIR.mkdir(exist_ok=True)

# Detect quasars with mixed WCS solutions across their FLT files.
# A quasar needs TweakReg if its exposures don't all share the same WCSNAME —
# different visits may have been aligned to different reference catalogs (or none),
# meaning they won't stack cleanly without first being brought to a common frame.
from astropy.io import fits

tweakreg_targets = {}  # {target_name: [flt_path, ...]}

for quasar_dir in sorted(RAW_DIR.iterdir()):
    flt_files = sorted(quasar_dir.glob('*flt.fits'))
    if not flt_files:
        continue

    wcsnames = [fits.getval(str(f), 'WCSNAME', ext=1) for f in flt_files]
    unique = set(wcsnames)

    if len(unique) > 1:
        tweakreg_targets[quasar_dir.name] = [str(f) for f in flt_files]
        print(f'NEEDS TWEAKREG: {quasar_dir.name}')
        for name, count in sorted((n, wcsnames.count(n)) for n in unique):
            print(f'  {count:2d} files — {name}')
    else:
        print(f'OK: {quasar_dir.name} — {next(iter(unique))}')

print(f'\n{len(tweakreg_targets)} quasar(s) flagged for TweakReg.')

In [ ]:
from astropy.coordinates import SkyCoord
from astropy import units as u
from astroquery.sdss import SDSS
from astroquery.gaia import Gaia
from astropy.units import Quantity

# Identify sky coordinates for each flagged quasar from the FLT primary header.
# Equivalent to table['ra_targ'][0] / table['dec_targ'][0] in the reference notebook —
# RA_TARG and DEC_TARG are written to ext=0 by CALWF3 and are the same for all
# exposures of a given target.
target_coords = {}

for target, flt_files in tweakreg_targets.items():
    ra  = fits.getval(flt_files[0], 'RA_TARG',  ext=0)
    dec = fits.getval(flt_files[0], 'DEC_TARG', ext=0)
    target_coords[target] = SkyCoord(ra=ra, dec=dec, unit=(u.deg, u.deg))
    print(f'{target}: RA={ra:.6f}, Dec={dec:.6f}')

In [ ]:
radius_sdss = Quantity(3., u.arcmin)

sdss_catalogs = {}  # {target: Table or None}

for target, flt_files in tweakreg_targets.items():
    coord = target_coords[target]
    print(f'\n{target}')

    sdss_query = SDSS.query_region(coord, radius=radius_sdss, spectro=False,
                                   fields=['ra', 'dec', 'g'])

    if sdss_query is None:
        print('  No SDSS sources returned — field may be outside SDSS footprint or too sparse.')
        sdss_catalogs[target] = None
        continue

    # Some versions of astroquery prepend an objID column — remove it if present.
    try:
        sdss_query.remove_column('objID')
    except KeyError:
        print('  No objID column found... proceeding.')

    catfile = f'{target}_sdss.cat'
    sdss_query.write(catfile, format='ascii.commented_header', overwrite=True)
    sdss_catalogs[target] = sdss_query
    print(f'  {len(sdss_query)} sources → {catfile}')
    print(sdss_query)

In [ ]:
for target, sdss_query in sdss_catalogs.items():
    if sdss_query is None:
        continue
    catfile = str(TWEAKREG_DIR / f'{target}_sdss.cat')
    sdss_query.write(catfile, format='ascii.commented_header', overwrite=True)
    print(f'{target} → {catfile}')

In [ ]:
radius_gaia = Quantity(5., u.arcmin)

gaia_catalogs = {}  # {target: Table}

for target, flt_files in tweakreg_targets.items():
    coord = target_coords[target]
    print(f'{target}')

    gaia_query = Gaia.query_object_async(coordinate=coord, radius=radius_gaia)
    gaia_catalogs[target] = gaia_query
    print(f'  {len(gaia_query)} Gaia sources within 5 arcmin')
    print(gaia_query)

In [ ]:
gaia_reduced = {}  # {target: reduced Table}

for target, gaia_query in gaia_catalogs.items():
    reduced_query = gaia_query['ra', 'dec', 'phot_g_mean_mag']
    gaia_reduced[target] = reduced_query
    print(f'{target}: {len(reduced_query)} sources')
    print(reduced_query)

In [ ]:
for target, reduced_query in gaia_reduced.items():
    catfile = str(TWEAKREG_DIR / f'{target}_gaia.cat')
    reduced_query.write(catfile, format='ascii.commented_header', overwrite=True)
    print(f'{target} → {catfile}')

In [ ]:
for target, flt_files in tweakreg_targets.items():
    refcat  = str(TWEAKREG_DIR / f'{target}_gaia.cat')
    wcsname = 'Gaia'

    # conv_width: 2x PSF FWHM — 2.5 for WFC3/IR (reference uses 3.5 for ACS/WFC or WFC3/UVIS)
    cw = 2.5

    # threshold: source detection level in counts above background.
    # Previous run at 200 found only 5-8 sources per image — far below the 15-source minimum.
    # Start at 30 for these sparse quasar fields; adjust after inspecting the shift file.
    threshold = 30.

    # searchrad: match radius in arcseconds between image and catalog sources.
    # Reference uses 0.1" because their images have tight Gaia pipeline WCS.
    # Our IDC-only files have no sky reference and may be offset by several arcseconds,
    # so we start conservatively at 1.0" and tighten if the fit converges well.
    searchrad = 1.0

    print(f'Running TweakReg (Gaia) for {target}...')
    tweakreg.TweakReg(
        flt_files,
        imagefindcfg={'threshold': threshold, 'conv_width': cw},
        refcat=refcat,
        interactive=False,
        see2dplot=False,
        shiftfile=True,
        outshifts=str(TWEAKREG_DIR / f'{target}_Gaia_shifts.txt'),
        logfile=str(TWEAKREG_DIR / f'{target}_tweakreg.log'),
        wcsname=wcsname,
        reusename=True,
        ylimit=0.3,
        fitgeometry='rscale',
        searchrad=searchrad,
        writecat=False,
        updatehdr=False,
    )
    clear_output()
    print(f'{target} done. Outputs in {TWEAKREG_DIR}')